# XAD quickstart: gradients in Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/auto-differentiation/xad-py/blob/main/notebooks/01_quickstart.ipynb)

[XAD](https://github.com/auto-differentiation/xad) is a fast automatic differentiation (AD) library for C++ with
Python bindings. This notebook shows the two first-order modes on a small function and checks the result against
finite differences:

1. **Adjoint (reverse) mode**: the whole gradient in one forward pass plus one backward pass.
2. **Forward (tangent) mode**: one directional derivative per pass.
3. **Check** against finite differences.

It runs top to bottom in a few seconds.

In [1]:
%pip install -q xad

Note: you may need to restart the kernel to use updated packages.


## A function to differentiate

XAD works by operator overloading: you write ordinary Python code, and when the inputs are XAD `Real` numbers the
operations are tracked. Use `xad.math` for math functions; it accepts both plain floats and XAD types, so the same
function serves for normal evaluation and for differentiation.

In [2]:
from xad import math as xm


def f(x0, x1, x2, x3):
    return xm.sin(x0) * xm.exp(x1) + x2 ** 2 / x3 - xm.log(1.0 + x0 * x3)


x = [1.0, 0.5, 2.0, 3.0]
print(f"f(x) = {f(*x):.10f}")

f(x) = 1.3343900835


## 1. Adjoint mode: all derivatives in one backward pass

Adjoint mode records the operations on a *tape*, then runs through it backwards once. The cost of the full gradient
is a small constant multiple of one function evaluation, **however many inputs there are**.

In [3]:
import xad.adj_1st as xadj

x_ad = [xadj.Real(v) for v in x]

with xadj.Tape() as tape:
    tape.registerInputs(x_ad)   # mark the independent variables
    tape.newRecording()         # start recording operations

    y = f(*x_ad)                # run the unchanged function

    tape.registerOutput(y)
    y.derivative = 1.0          # seed dy/dy = 1
    tape.computeAdjoints()      # one backward pass

    grad_adj = [xi.derivative for xi in x_ad]

print(f"y = {y.value:.10f}")
for i, g in enumerate(grad_adj):
    print(f"dy/dx{i} = {g:+.10f}")

y = 1.3343900835
dy/dx0 = +0.1408079043
dy/dx1 = +1.3873511113
dy/dx2 = +1.3333333333
dy/dx3 = -0.6944444444


## 2. Forward mode: one direction per pass

Forward mode carries a derivative alongside each value. No tape is needed. Seeding input `i` with derivative 1 gives
`dy/dx_i`, so a full gradient needs one pass per input. Forward mode is the right choice when there are few inputs
and many outputs.

In [4]:
import xad.fwd_1st as xfwd

grad_fwd = []
for i in range(len(x)):
    x_fwd = [xfwd.Real(v) for v in x]
    x_fwd[i].derivative = 1.0   # seed direction e_i
    y = f(*x_fwd)
    grad_fwd.append(y.derivative)

for i, g in enumerate(grad_fwd):
    print(f"dy/dx{i} = {g:+.10f}")

dy/dx0 = +0.1408079043
dy/dx1 = +1.3873511113
dy/dx2 = +1.3333333333
dy/dx3 = -0.6944444444


## 3. Check against finite differences

Central differences need two extra evaluations per input and are only accurate to about `h**2`. AD derivatives are
exact up to floating-point rounding.

In [5]:
h = 1e-6
grad_fd = []
for i in range(len(x)):
    x_up, x_dn = list(x), list(x)
    x_up[i] += h
    x_dn[i] -= h
    grad_fd.append((f(*x_up) - f(*x_dn)) / (2 * h))

print(f"{'':6} {'adjoint':>16} {'forward':>16} {'finite diff':>16} {'|adj - fd|':>10}")
for i in range(len(x)):
    print(f"dy/dx{i} {grad_adj[i]:16.10f} {grad_fwd[i]:16.10f} {grad_fd[i]:16.10f} "
          f"{abs(grad_adj[i] - grad_fd[i]):10.1e}")

                adjoint          forward      finite diff |adj - fd|
dy/dx0     0.1408079043     0.1408079043     0.1408079043    4.3e-11
dy/dx1     1.3873511113     1.3873511113     1.3873511113    3.7e-12
dy/dx2     1.3333333333     1.3333333333     1.3333333333    3.6e-11
dy/dx3    -0.6944444444    -0.6944444444    -0.6944444444    5.1e-11


Adjoint and forward mode agree to machine precision; finite differences agree only to about 1e-10, the
truncation and rounding error of the bump.

## Next steps

- [Black-Scholes and Monte Carlo Greeks](https://colab.research.google.com/github/auto-differentiation/xad-py/blob/main/notebooks/02_option_greeks.ipynb): all sensitivities of an option price in one adjoint pass.
- [Adjoint vs finite differences at scale](https://colab.research.google.com/github/auto-differentiation/xad-py/blob/main/notebooks/03_adjoint_vs_finite_differences.ipynb): gradient of a PDE model with hundreds of parameters.
- [Python documentation](https://auto-differentiation.github.io/tutorials/python) and the [XAD C++ library](https://github.com/auto-differentiation/xad).